In [3]:
import pandas as pd

log = pd.read_csv('/Users/francescameneghello/Downloads/simulated_log_test_only_sepsis_0.1.csv')

In [4]:
log

In [6]:
log['resource'].unique().tolist()

In [7]:
def start_error(log):
    error = 0
    log['end_time'] = pd.to_datetime(log['end_time'])
    log['start_time'] = pd.to_datetime(log['start_time'])
    for index, row in log.iterrows():
        if row['start_time']>row['end_time']:
            error += 1
    return error

In [8]:
start_error(log)

In [11]:
### overlap

def overlap_error(log):
    log = log[log['resource'].notna()]
    res = log['resource'].unique().tolist()
    capacity = {r: 1 for r in res}
    overlap_error = 0
    resource_work = {r: {} for r in res}
    log['end_time'] = pd.to_datetime(log['end_time'])
    log['start_time'] = pd.to_datetime(log['start_time'])
    for index, row in log.iterrows():
        start = int(row['start_time'].timestamp())
        end = int(row['end_time'].timestamp())
        for i in range(start, end):
            if i in resource_work[row['resource']]:
                resource_work[row['resource']][i] += 1
                if resource_work[row['resource']][i] > capacity[row['resource']]:
                    overlap_error += 1
            else:
                resource_work[row['resource']][i] = 1
    return overlap_error

In [3]:
log = pd.read_csv('/Users/francescameneghello/Documents/GitHub/nirdizati-light/predictive_models_processing_time/sepsis_estimated_start.csv', sep=";")
log

In [33]:
import pandas as pd

log = pd.read_csv('/Users/francescameneghello/Documents/GitHub/nirdizati-light/predictive_models_processing_time/BPI_Challenge_2012_estimated_start.csv', sep=",")
start_times = []
log['start:timestamp'] = log['start:timestamp'].astype(str).str[:19]
log['start:timestamp'] = pd.to_datetime(log['start:timestamp'], utc=True, format="%Y-%m-%d %H:%M:%S")
caseid_unique = list(log['caseid'].unique())
for caseid in caseid_unique:
    group_case = log[log['caseid'] == caseid].sort_values(by='start:timestamp')
    start_times.append(group_case.iloc[0]['start:timestamp'])

In [28]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy import stats

# Example list of datetime objects (assumed to be defined elsewhere)
datetimes = sorted(start_times)

# Convert datetimes to numeric values (e.g., seconds since the first datetime)
time_deltas = np.array([
    (datetimes[i] - datetimes[i - 1]).total_seconds()
    for i in range(1, len(datetimes))
])

# --- Outlier removal using Z-score ---
z_scores = np.abs(stats.zscore(time_deltas))
threshold = 3
filtered_deltas = time_deltas[z_scores < threshold]

# List of distributions to test
distributions = ['norm', 'expon', 'lognorm', 'gamma', 'beta']
results = []

# Fit each distribution and compute the Kolmogorov-Smirnov (KS) statistic
x = np.linspace(min(filtered_deltas), max(filtered_deltas), 100)
hist_vals, bin_edges = np.histogram(filtered_deltas, bins=5, density=True)

for dist_name in distributions:
    dist = getattr(stats, dist_name)
    params = dist.fit(filtered_deltas)
    ks_stat, ks_pvalue = stats.kstest(filtered_deltas, dist_name, args=params)
    
    results.append({
        'distribution': dist_name,
        'params': params,
        'ks_stat': ks_stat,
        'ks_pvalue': ks_pvalue
    })

# Sort by KS statistic (lower is better)
results.sort(key=lambda r: r['ks_stat'])

# Plot the histogram and best 3 fitted PDFs
plt.hist(filtered_deltas, bins=5, density=True, alpha=0.5, label='Data histogram')

for result in results[:3]:
    dist = getattr(stats, result['distribution'])
    pdf = dist.pdf(x, *result['params'])
    plt.plot(x, pdf, label=f"{result['distribution']}")

plt.xlabel('Seconds since start')
plt.ylabel('Density')
plt.legend()
plt.title("Top 3 Fitted Distributions (Outliers Removed)")
plt.tight_layout()

results[:3]  # Show the best 3 fits


In [27]:
mu = 20203198.389895137/3600
sigma = 10528412.75221794/3600  # standard deviation
n_samples = 1000            # number of data points

# Generate samples
data = np.random.normal(loc=mu, scale=sigma, size=10)
data

In [29]:
36028/3600